#### GROUP 17: Shuvojyoti Singha, Artjom Smorgulenko, Kaan Özkiliç, Samuel Pasierb

# Exploring Unsupervised Learning with the k-Means Algorithm

## Dataset Background
> This dataset contains the answers to 60 questions to determine the personality of a person. The possible personalities are: 
*  ESTJ - The Supervisor
*  ENTJ - The Commander 
*  ESFJ - The Provider 
*  ENFJ - The Giver
*  ISTJ - The Inspector
*  ISFJ - The Nurturer 
*  INTJ - The Mastermind
*  INFJ - The Counselor
*  ESTP - The Doer
*  ESFP - The Performer
*  ENTP - The Visionary
*  ENFP - The Champion 
*  ISTP - The Craftsman
*  ISFP - The Composer 
*  INTP - The Thinker 
*  INFP - The Idealist 
> The answers range from -3 to 3 depending on much the person agrees with the statement. -3 means full disagreement and 3 means full agreement. 

## Imports


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from matplotlib import pyplot as plt
from enum import Enum

## Constants & Variables


In [ ]:
class Personality:
    full_name: str
    code: str
    index: int
    colour: tuple[float]
    
    def __init__(self, full_name: str, code: str, index: int, colour: tuple[float]) -> None:
        self.full_name = full_name
        self.code = code
        self.index = index
        self.colour = tuple(colour[i] / 255 for i in range(3))


class Personalities(Enum):
    ESTJ = Personality("The Supervisor", "ESTJ", 0, (233, 30, 99))
    ISTJ = Personality("The Inspector", "ISTJ", 4, (226, 46, 210))
    ESTP = Personality("The Doer", "ESTP", 8, (255, 87, 34))
    ISTP = Personality("The Craftsman", "ISTP", 12, (244, 67, 54))
    ESFP = Personality("The Performer", "ESFP", 9, (255, 152, 0))
    ESFJ = Personality("The Provider", "ESFJ", 2, (121, 85, 72))
    ISFP = Personality("The Composer", "ISFP", 13, (255, 235, 59))
    ISFJ = Personality("The Nurturer", "ISFJ", 5, (255, 193, 7))
    INFP = Personality("The Idealist", "INFP", 15, (205, 220, 57))
    INFJ = Personality("The Counselor", "INFJ", 7, (139, 195, 74))
    ENFP = Personality("The Champion", "ENFP", 11, (0, 150, 136))
    ENFJ = Personality("The Giver", "ENFJ", 3, (76, 175, 80))
    INTJ = Personality("The Mastermind", "INTJ", 6, (156, 39, 176))
    INTP = Personality("The Thinker", "INTP", 14, (0, 188, 212))
    ENTJ = Personality("The Commander", "ENTJ", 1, (63, 81, 181))
    ENTP = Personality("The Visionary", "ENTP", 10, (33, 150, 243))
    

    def from_index(index) -> Personality:
        for personality in Personalities:
            if personality.value.index == index: return personality.value
    

class Question:
    index: int
    text: str
    
    def __init__(self, index, text) -> None:
        self.index = index
        self.text = text
        

questions: list[Question] = []
pca: PCA = PCA(2)

## Loading dataset
This dataset is downloaded from Kaggle. It includeds 62 columns, 1 of them being an ID and 1 the Personality, the rest are questions. There was � in the .csv file which had to be replaced with ' manually.

[The link to the dataset on Kaggle.](https://www.kaggle.com/datasets/anshulmehtakaggl/60k-responses-of-16-personalities-test-mbt)

In [ ]:
df: pd.DataFrame = pd.read_csv('personality-test-dataset.csv')
df.head()

## Data Cleaning
The missing data is handled and questions are replaced with "Question x" to make the dateset look nicer.

- The missing values are filled with 0.
- ID column is dropped because it is unnecesarry.
- Personiality column (target value) is dropped because it is not needed for clustering
- Questions are replaced with "Question x"

In [ ]:
df.fillna(0, inplace=True)
print(df.isnull().sum())

In [ ]:
correct_personality: pd.Series = df.Personality

if "Response Id" in df.columns: df = df.drop(["Response Id"], axis=1)
if "Personality" in df.columns: df = df.drop(["Personality"], axis=1)

print(df.columns)

In [ ]:
questions = [Question(i + 1, q) for i, q in enumerate(df.columns)]
df.columns = [f"Questions {q.index}" for q in questions]

In [ ]:
df.head()

## Algorithm Implementation

### 1. Transforming the data from 60 Dimensions to 2 Dimensions

In [ ]:
df = pd.DataFrame(data=pca.fit_transform(df), columns=["A", "B"])
df.head()

# df: pd.DataFrame = pd.DataFrame(np.array(list(np.array([sum(list(filter(lambda x: x > 0, row))), sum(list(filter(lambda x: x < 0, row)))]) for row in df.to_numpy())), columns=["Positive", "Negative"])
# df.head()

### 2. Run k-Means Algorithm

In [ ]:
km = KMeans(n_clusters=16)
clusters: np.ndarray = km.fit_predict(df)
df["Personality"] = clusters

### 3. Divide the dataframe into 16 dataframes based on personalities for visualization

In [ ]:
dataframes: list[pd.DataFrame] = []

for personality in Personalities:
    dataframes.append(df[df.Personality == personality.value.index])

## Visualization

In [ ]:
plt.figure(figsize=(13, 13))

for dataframe in dataframes:
    if len(dataframe) == 0: continue
    personality: Personality = Personalities.from_index(dataframe.Personality.values[0])
    plt.scatter(dataframe["A"], dataframe["B"], c=[personality.colour], label=f"{personality.code} - {personality.full_name}")

plt.title("16 Personalities")
plt.ylabel("PCA Y")
plt.xlabel("PCA X")
plt.gca().set_facecolor("black")
plt.legend()
plt.show()

## Elbow Method Implementation

In [ ]:
inertia = []
k_range = range(1, 21)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(df)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(k_range, inertia, marker='o')
plt.xticks(k_range)
plt.xlabel("number of clusters (k)")
plt.ylabel("inertia")
plt.title("Elbow method with different k values")
plt.show()


## Real World Applications for k-means Algorithm

- **Document classification**: 
k-means algorithm is used to cluster and organize documents that are unstructured. After converting each document to a vector representation, term frequency is used to identify commonly used terms to help classify each document. For this purpose, k-means is a highly suitable algorithm.

- **Anomaly/Outlier Detection**:
k-means can be used to identify outliers/anomalies in a dataset, which is important in terms of fault detection. Data points that are away from any centroid could often be flagged as a potential fault on the dataset. This is also quite significant in cybersecurity field.